# Image Segmentation using LiteRT

This notebook demonstrates how to use LiteRT for image segmentation. It loads an image, processes it with a segmentation model, and saves the resulting segmentation mask and blended output image.

## Install Required Packages

First, let's install the LiteRT package from the local wheel file.

In [1]:
# Install the LiteRT package from the local wheel file
import sys
import subprocess
import os

# Path to the wheel file
wheel_file = "./ai_edge_litert-1.2.0-cp312-cp312-macosx_12_0_arm64.whl"

# Check if the file exists
if not os.path.exists(wheel_file):
    print(f"Error: Wheel file {wheel_file} not found.")
else:
    # Get absolute path to the wheel file
    wheel_abs_path = os.path.abspath(wheel_file)
    print(f"Installing wheel from: {wheel_abs_path}")
    
    # Install the wheel file
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall" ,wheel_abs_path, "--no-cache-dir"])
    print("Installation complete.")

Installing wheel from: /Users/jj/codespace/LiteRT_749469609/litert/samples/image_segmentation/python/ai_edge_litert-1.2.0-cp312-cp312-macosx_12_0_arm64.whl
Processing ./ai_edge_litert-1.2.0-cp312-cp312-macosx_12_0_arm64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 74.4 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.2.10
    Uninstalling flatbuffers-25.2.10:
      Successfully uninstalled flatbuffers-25.2.10
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
  Attempting uninstall: ai-edge-litert
    Found existing installation: ai-edge-litert 1.2.0
    Uninstalling ai-edge-litert-1.2.0:
      Successfully uninstalled ai-edge-litert-1.2.0
Installation complete.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
executorch 0.5.0 requires numpy==2.0.0; python_version >= "3.10", but you have numpy 2.2.5 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.1
[notice] To update, run: pip install --upgrade pip


## Import Required Libraries

In [2]:
import os
import random
import time
from typing import Any, Dict, List, Tuple

import numpy as np
from PIL import Image

# Import CompiledModel from ai_edge_litert
from ai_edge_litert.compiled_model import CompiledModel

ImportError: dlopen(/Users/jj/.local/share/mise/installs/python/3.12.8/lib/python3.12/site-packages/ai_edge_litert/_pywrap_litert_compiled_model_wrapper.so, 0x0002): Library not loaded: @rpath/libtensorflow_framework.2.dylib
  Referenced from: <05B8207C-EC6A-350C-9116-CE9DCB6D9694> /Users/jj/.local/share/mise/installs/python/3.12.8/lib/python3.12/site-packages/ai_edge_litert/_pywrap_litert_compiled_model_wrapper.so
  Reason: tried: '/Users/jj/.local/share/mise/installs/python/3.12.8/lib/python3.12/site-packages/ai_edge_litert/../../../../_solib_darwin_arm64/_U_S_Slitert_Spython_Slitert_Uwrapper_Scompiled_Umodel_Uwrapper_C_Upywrap_Ulitert_Ucompiled_Umodel_Uwrapper.so___Uexternal_Sorg_Utensorflow_Stensorflow/libtensorflow_framework.2.dylib' (no such file), '/Users/jj/.local/share/mise/installs/python/3.12.8/lib/python3.12/site-packages/ai_edge_litert/_pywrap_litert_compiled_model_wrapper.so.runfiles/litert/_solib_darwin_arm64/_U_S_Slitert_Spython_Slitert_Uwrapper_Scompiled_Umodel_Uwrappe

## Define Helper Functions for File Loading

We'll replace the resource_loader with standard Python file handling.

In [ ]:
def get_path_to_file(file_path: str) -> str:
    """Get the absolute path to a file.
    
    This function replaces resource_loader.get_path_to_datafile.
    
    Args:
        file_path: Path to the file, relative to the current directory
        
    Returns:
        Absolute path to the file
        
    Raises:
        FileNotFoundError: If the file doesn't exist
    """
    # Check if the path is already absolute
    if os.path.isabs(file_path):
        full_path = file_path
    else:
        # Convert relative path to absolute path
        full_path = os.path.abspath(file_path)
    
    # Check if the file exists
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"File not found: {full_path}")
    
    return full_path

## Image Segmentation Helper Class

In [ ]:
class ImageSegmentation:
  """Helper class for image segmentation tasks using LiteRT.

  Provides methods to initialize a segmentation model, preprocess images,
  run inference, and process the segmentation masks.
  """

  def __init__(self, model_path: str):
    """Initialize the segmentation helper.

    Args:
        model_path: Path to the TFLite model file for segmentation
    """
    self.model_path = model_path
    self.model = None
    self.colored_labels = self._create_colored_labels()
    self.input_size = (256, 256)
    self.initialize()

  def initialize(self) -> None:
    """Initialize the CompiledModel for segmentation."""
    try:
      self.model = CompiledModel.from_file(self.model_path)
      print(
          "Model loaded successfully with"
          f" {self.model.get_num_signatures()} signatures"
      )
    except Exception as e:
      print(f"Failed to create LiteRT model: {str(e)}")
      raise

  def normalize(
      self, image: np.ndarray, mean: float = 127.5, stddev: float = 127.5
  ) -> np.ndarray:
    """Normalize the input image for the model.

    Args:
        image: Input image as numpy array (H, W, 3)
        mean: Mean value for normalization
        stddev: Standard deviation for normalization

    Returns:
        Normalized image as a numpy array
    """
    # Ensure image is in the right format (float32)
    image = image.astype(np.float32)

    # Normalize the image
    normalized = (image - mean) / stddev

    # Ensure the result is contiguous in memory and in float32 format
    return np.ascontiguousarray(normalized, dtype=np.float32)

  def preprocess_image(
      self, image_path: str, rotation_degrees: int = 0
  ) -> Tuple[np.ndarray, np.ndarray]:
    """Preprocess an image for segmentation.

    Args:
        image_path: Path to the input image
        rotation_degrees: Optional rotation in degrees

    Returns:
        Tuple of (original image, preprocessed image)
    """
    # Load the image
    image = Image.open(image_path)

    # Resize to expected input dimensions
    width, height = self.input_size
    resized_image = image.resize((width, height))

    # Apply rotation if needed
    if rotation_degrees != 0:
      resized_image = resized_image.rotate(-rotation_degrees)

    # Convert to numpy array
    img_array = np.array(resized_image)

    # Normalize the image
    normalized = self.normalize(img_array)

    return np.array(image), normalized

  def segment(self, image: np.ndarray) -> Tuple[np.ndarray, float]:
    """Perform segmentation on a preprocessed image.

    Args:
        image: Preprocessed image as numpy array

    Returns:
        Tuple of (segmentation mask, inference time in ms)

    Raises:
        RuntimeError: If the model creation failed
    """
    if self.model is None:
      raise RuntimeError("Model not compiled.")

    # Track inference time
    start_time = time.time()

    # Create input and output buffers
    sig_idx = 0
    input_buffers = self.model.create_input_buffers(sig_idx)
    output_buffers = self.model.create_output_buffers(sig_idx)

    # Write the preprocessed image to the input buffer
    # Reshape to match the expected input format
    input_data = image.reshape(-1)
    input_buffers[0].write(input_data)

    # Run inference
    self.model.run_by_index(sig_idx, input_buffers, output_buffers)

    # Get output data (segmentation logits)
    # Assuming the output has shape [height, width, num_classes]
    height, width = self.input_size
    num_classes = 6

    # Read output data
    output_size = height * width * num_classes
    output_data = output_buffers[0].read(output_size, np.float32)
    output_data = output_data.reshape(height, width, num_classes)

    # Process output to get segmentation mask
    mask = self._process_output(output_data)

    # Calculate inference time
    inference_time = (
        time.time() - start_time
    ) * 1000  # Convert to milliseconds

    # Clean up
    for buf in input_buffers:
      buf.destroy()
    for buf in output_buffers:
      buf.destroy()

    return mask, inference_time

  def _process_output(self, output_data: np.ndarray) -> np.ndarray:
    """Process model output to create a segmentation mask.

    Args:
        output_data: Model output as numpy array [height, width, num_classes]

    Returns:
        Segmentation mask as numpy array
    """
    # Find the class with the highest probability for each pixel
    mask = np.argmax(output_data, axis=2).astype(np.uint8)

    return mask

  def _create_colored_labels(self) -> List[Dict[str, Any]]:
    """Create colored labels for visualization.

    Returns:
        List of dictionaries containing label information and colors
    """
    labels = [
        "background",
        "aeroplane",
        "bicycle",
        "bird",
        "boat",
        "bottle",
        "bus",
        "car",
        "cat",
        "chair",
        "cow",
        "dining table",
        "dog",
        "horse",
        "motorbike",
        "person",
        "potted plant",
        "sheep",
        "sofa",
        "train",
        "tv",
        "------",
    ]

    # Create a list of colored labels
    colored_labels = []

    # Generate visually distinct colors using golden ratio
    golden_ratio_conjugate = 0.618033988749895
    hue = random.random()

    for idx, label in enumerate(labels):
      if idx == 0:
        # Background is black
        color = (0, 0, 0)
      else:
        # Generate colors using golden ratio method
        hue += golden_ratio_conjugate
        hue %= 1.0

        # Convert HSV to RGB
        h = hue * 360
        s = 0.7
        v = 0.8

        # HSV to RGB conversion
        c = v * s
        x = c * (1 - abs((h / 60) % 2 - 1))
        m = v - c

        if h < 60:
          r, g, b = c, x, 0
        elif h < 120:
          r, g, b = x, c, 0
        elif h < 180:
          r, g, b = 0, c, x
        elif h < 240:
          r, g, b = 0, x, c
        elif h < 300:
          r, g, b = x, 0, c
        else:
          r, g, b = c, 0, x

        r, g, b = int((r + m) * 255), int((g + m) * 255), int((b + m) * 255)
        color = (r, g, b)

      colored_labels.append(
          {"label": label, "display_name": label, "color": color}
      )

    return colored_labels

  def create_colored_mask(self, mask: np.ndarray) -> np.ndarray:
    """Create a colored segmentation mask for visualization.

    Args:
        mask: Segmentation mask as numpy array

    Returns:
        Colored mask as numpy array
    """
    height, width = mask.shape
    colored_mask = np.zeros((height, width, 3), dtype=np.uint8)

    # Apply colors to each class
    for label_idx, label_info in enumerate(self.colored_labels):
      if label_idx >= len(self.colored_labels):
        continue

      # Create a mask for this class
      class_mask = mask == label_idx

      # Apply color to this class
      color = label_info["color"]
      colored_mask[class_mask] = color

    return colored_mask

  def blend_images(
      self, image: np.ndarray, colored_mask: np.ndarray, alpha: float = 0.5
  ) -> np.ndarray:
    """Blend the original image with the colored segmentation mask.

    Args:
        image: Original image as numpy array
        colored_mask: Colored segmentation mask as numpy array
        alpha: Blending factor (0.0 to 1.0)

    Returns:
        Blended image as numpy array
    """
    # Resize colored mask to match original image if necessary
    if image.shape[:2] != colored_mask.shape[:2]:
      colored_mask_pil = Image.fromarray(colored_mask)
      colored_mask_pil = colored_mask_pil.resize(
          (image.shape[1], image.shape[0])
      )
      colored_mask = np.array(colored_mask_pil)

    # Blend images
    blended = (image * (1 - alpha) + colored_mask * alpha).astype(np.uint8)

    return blended

## Run Image Segmentation

Now let's run the image segmentation on a sample image.

In [ ]:
# Update these paths to your desired locations
model_path = "testdata/selfie_multiclass_256x256.tflite"  # Update this with your model path
image_path = "testdata/image.jpg"
output_dir = "./output"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

try:
    # Get absolute paths to files
    model_path = get_path_to_file(model_path)
    image_path = get_path_to_file(image_path)
    
    print(f"Using model: {model_path}")
    print(f"Using image: {image_path}")
    
    # Initialize the segmentation helper
    segmentation = ImageSegmentation(model_path)
    
    # Preprocess the image
    original_image, preprocessed_image = segmentation.preprocess_image(image_path)
    
    # Run segmentation
    mask, inference_time = segmentation.segment(preprocessed_image)
    
    print(f"Segmentation finished in {inference_time:.2f} ms")
    
    # Create a colored mask
    colored_mask = segmentation.create_colored_mask(mask)
    
    # Blend with original image
    blended_image = segmentation.blend_images(original_image, colored_mask)
    
    # Save results
    output_base = os.path.splitext(os.path.basename(image_path))[0]
    
    # Save mask as image
    mask_image = Image.fromarray(colored_mask)
    mask_path = os.path.join(output_dir, f"{output_base}_mask.png")
    mask_image.save(mask_path)
    
    # Save blended image
    blended_image_pil = Image.fromarray(blended_image)
    blended_path = os.path.join(output_dir, f"{output_base}_blended.png")
    blended_image_pil.save(blended_path)
    
    print(f"Results saved to: {output_dir}")
    print(f"Mask: {mask_path}")
    print(f"Blended image: {blended_path}")
    
    # Display the results
    from IPython.display import display
    from PIL import Image
    
    print("Original Image:")
    display(Image.fromarray(original_image))
    
    print("Segmentation Mask:")
    display(mask_image)
    
    print("Blended Result:")
    display(blended_image_pil)
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please update the paths to your model and image files.")
except Exception as e:
    print(f"Error: {e}")

## Visualize Results with Matplotlib (Alternative)

If you prefer to visualize the results using matplotlib, you can use this cell instead.

In [ ]:
import matplotlib.pyplot as plt

# Display the results if available
try:
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.title("Original Image")
    plt.imshow(original_image)
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.title("Segmentation Mask")
    plt.imshow(colored_mask)
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.title("Blended Result")
    plt.imshow(blended_image)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
except NameError:
    print("Run the previous cell first to generate the segmentation results.")